### Tools

Models can request  to call tools that perform tasks such as fetching data from db, searching web, or running code. Tools are pairings of:
- A schema, including name of the tool, a desc, and/or arguments def (often JSON)
 - A function or coroutine to execute

In [3]:
from langchain.chat_models import init_chat_model 

### Google Gemini
model = init_chat_model("google_genai:gemini-3.5-flash-lite")
# response = model.invoke("Why do birds sing?")
# response

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a location"""
    return f"It is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("Whats the weather in Coimbatore?")
print(response)

for tool_call in response.tool_calls:
    #view the tool call model made

    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Coimbatore"}'}, '__gemini_function_call_thought_signatures__': {'call_907729941884574149': 'EmAKXgFpFH0T7cF2tgGfJS6gA7D8mjHC+iCpGGzgkaMZFmHd/wxAsDnttZk5WvcAAEwrz5dK5I4hjHzAmj++VwhDnKMoQrhwF8jgiyOwCWArPU5NOMDBJohaabRcywzJZ8g='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1df62-8998-7380-9c7c-4f9688a0efd5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Coimbatore'}, 'id': 'call_907729941884574149', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 17, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Args: {'location': 'Coimbatore'}


### Tool Execution Loops

In [8]:
# Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in texas?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: pass results back to model for final response
final_repsonse =  model_with_tools.invoke(messages)
print(final_repsonse.text)

The weather in Texas is currently sunny.


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in texas?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Texas"}'}, '__gemini_function_call_thought_signatures__': {'call_2405436673493747263': 'EmAKXgFpFH0TdzBt9Vw+3OMDGTZgQ8oAAufUuFMluAHG5ht/UjEZ1mmV3YeiIjsLsIFe77oIlqik36WXAiWsLPvi8HAEp7dJgRVwxEFxwQt/9VaBHr/eRHgu3lmHSWc6xFs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1df67-ba54-77fe-9242-1d944aca3f0a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Texas'}, 'id': 'call_2405436673493747263', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='It is sunny in Texas', name='get_weather', tool_call_id='call_2405436673493747263')]